# Set Up

In [1]:
!pip install groq transformers --quiet

In [1]:
import pandas as pd
import numpy as np

## Load The data

In [2]:
### UPLOAD THE strategyqa_testset JSON FILE TO COLAB
df = pd.read_json("strategyqa_testset_50.json")
df.head()

,qid,question,answer
0,a651ba82c5e39990d737,Do the directors of The Matrix advocate for tr...,True
1,66b3cfaa499773bdf513,Would it be difficult for Will Ferrell to win ...,True
2,a5f8af1dd0e9c46c47be,"Does Orange County, California require airplan...",True
3,b1e1256007b0a4a341a7,If someone loves buffalo wings do they enjoy c...,True
4,11d009721f27a60f9cff,Would a Pict be confused by Old English?,True


## Load the models


### Loading Llama using Groq API

In [ ]:
from groq import Groq

API_KEY = ""
client = Groq(api_key=API_KEY)

In [4]:
DEFAULT_SYSTEM_PROMPT = "You are a helpful asssitant. Please Answer only in True or False"

In [5]:
MODEL_NAME = "llama-3.1-8b-instant"

def query_llama(prompt, temperature,sys_prompt=DEFAULT_SYSTEM_PROMPT, max_tokens=512):
    try:
        completion = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "system", "content": sys_prompt},
                {"role": "user", "content": prompt}],
            temperature=temperature,
            max_tokens=max_tokens
        )

        return completion.choices[0].message.content

    except Exception as e:
        return f"Error: {str(e)}"

In [6]:
# TESTING API CONNECTION
print(query_llama(prompt="Is the sky blue?", temperature=0))


True.


### Loading GPT-2
We'll use it only for zero shot

In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer

gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")
gpt2_model = AutoModelForCausalLM.from_pretrained("gpt2")

def query_gpt(prompt, temprature, num_tokens):
    input_ids = gpt2_tokenizer.encode(prompt, return_tensors='pt')
    output = gpt2_model.generate(
        input_ids,
        max_length=num_tokens + len(input_ids[0]),
        num_return_sequences=1,
        do_sample= temprature > 0,
        temperature=temprature,
        top_k=50,
        pad_token_id=gpt2_tokenizer.eos_token_id
    )
    generated_text = gpt2_tokenizer.decode(output[0], skip_special_tokens=True)
    generated_text_without_prompt = generated_text[len(prompt):].strip()
    return generated_text_without_prompt

/home/bnet/adamfleisher/anaconda3/envs/nlp_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 2029.85it/s]


# Zero Shot


## Zero Shot with Llama

In [8]:
def zeroshot(question):
  temperatue = 0
  pred = query_llama(prompt=question, temperature=temperatue)
  return pred

In [9]:
df["zero_shot_pred"] = df["question"].apply(lambda q: zeroshot(q))

## Evaluating answers
Implement this function wisely and genericly - we will use it throught the exercise

In [10]:
def normalize_answer(text):
  if text is None:
    return None
  if isinstance(text, bool):
    return text

  s = str(text).strip().lower()
  for ch in '.,!?;:':
    s = s.replace(ch, ' ')
  s = ' '.join(s.split())

  if s in ('true', 'yes'):
    return True
  if s in ('false', 'no'):
    return False

  labels = []
  for word in s.split():
    if word in ('true', 'yes'):
      labels.append(True)
    elif word in ('false', 'no'):
      labels.append(False)

  return labels[-1] if labels else None


def evaluate(true_answer, model_pred):
  norm_true = normalize_answer(true_answer)
  norm_pred = normalize_answer(model_pred)
  if norm_true is None or norm_pred is None:
    return 0
  return int(norm_true == norm_pred)

In [11]:
zero_shot_acc = df.apply(lambda row: evaluate(row['answer'], row['zero_shot_pred']), axis=1).mean()
print(f"Zero Shot acc with Llama: {zero_shot_acc}")

Zero Shot acc with Llama: 0.68


## Zero Shot with GPT-2

In [12]:
def zeroshot_gpt(question):
  temprature = 0
  prompt = f"{DEFAULT_SYSTEM_PROMPT}\nQuestion: {question}\nAnswer:"
  pred = query_gpt(prompt, temprature, num_tokens=5)
  return pred

In [13]:
df["zero_shot_gpt_pred"] = df["question"].apply(lambda q: zeroshot_gpt(q))
zero_shot_acc = df.apply(lambda row: evaluate(row['answer'], row['zero_shot_gpt_pred']), axis=1).mean()
print(f"Zero Shot acc with GPT-2: {zero_shot_acc}")

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Zero Shot acc with GPT-2: 0.48


In [14]:
pred_col = "zero_shot_gpt_pred"  # use "zero_shot_pred" for Llama

for _, row in df.iterrows():
    if evaluate(row['answer'], row[pred_col]) == 0:
        print(f"Question: {row['question']}")
        print(f"  True answer:  {row['answer']}")
        print(f"  Model answer: {row[pred_col]}")
        print()

Question: Is tobacco use made to seem enjoyable in Alice's Adventures in Wonderland?
  True answer:  True
  Model answer: No.
Question:

Question: Is Brooklyn known for its bread products?
  True answer:  True
  Model answer: Brooklyn is known for its

Question: Could ten gallons of seawater crush a six year old?
  True answer:  True
  Model answer: No.
Question:

Question: Would students at Marist have to petition to get a rowing team?
  True answer:  False
  Model answer: Yes, students at Mar

Question: Does Rusev have to worry about human overpopulation in his homeland?
  True answer:  False
  Model answer: Yes, he does.

Question: Is Morocco an ideal location for water skiing?
  True answer:  False
  Model answer: Morocco is a great place

Question: Do white blood cells outnumber red blood cells in the human body?
  True answer:  False
  Model answer: Yes, white blood cells

Question: Would Achilles dominate Legolas in a hypothetical fight?
  True answer:  False
  Model answer: Yes

# Self-Consistency

**From now on, we will use only Llama (and NOT GPT-2)**

In [15]:
def self_consistency(question, num_sampling_iterations=5, temperatue=1, return_votes=False):
  samples = []
  for _ in range(num_sampling_iterations):
    response = query_llama(prompt=question, temperature=temperatue)
    norm = normalize_answer(response)
    if norm is not None:
      samples.append(norm)

  if not samples:
    fallback = query_llama(prompt=question, temperature=0)
    pred = normalize_answer(fallback)
    if pred is None:
      pred = fallback
    votes = {pred: 1} if pred is not None else {}
    if return_votes:
      return pred, votes, samples
    return pred

  votes = {}
  for label in samples:
    votes[label] = votes.get(label, 0) + 1

  max_count = max(votes.values())
  tied = {label for label, count in votes.items() if count == max_count}

  pred = None
  for label in samples:
    if label in tied:
      pred = label
      break

  if return_votes:
    return pred, votes, samples
  return pred

In [16]:
df["sc_pred"] = df["question"].apply(lambda q: self_consistency(q))
sc_acc = df.apply(lambda row: evaluate(row['answer'], row['sc_pred']), axis=1).mean()
print(f"Self-Consistnecy acc with Llama: {sc_acc}")

Self-Consistnecy acc with Llama: 0.72


## 1.3.c Temperature Analysis

Repeat self-consistency with temperatures 0, 0.5, and 1.0, and inspect the internal voting distributions.

In [17]:
from collections import Counter

temperatures = [0, 0.5, 1.0]
sc_temp_results = {}

for temp in temperatures:
  records = []

  for _, row in df.iterrows():
    pred, votes, samples = self_consistency(
      row['question'],
      num_sampling_iterations=5,
      temperatue=temp,
      return_votes=True,
    )
    true_votes = votes.get(True, 0)
    false_votes = votes.get(False, 0)
    records.append({
      'question': row['question'],
      'true_answer': row['answer'],
      'pred': pred,
      'votes': votes,
      'vote_pattern': (true_votes, false_votes),
      'correct': evaluate(row['answer'], pred),
      'unanimous': len(votes) <= 1,
      'split_vote': true_votes > 0 and false_votes > 0,
    })

  acc = sum(r['correct'] for r in records) / len(records)
  unanimous = sum(r['unanimous'] for r in records)
  split_votes = sum(r['split_vote'] for r in records)
  pattern_counter = Counter(r['vote_pattern'] for r in records)

  sc_temp_results[temp] = records

  print(f"\n=== Temperature = {temp} ===")
  print(f"Accuracy: {acc:.3f}")
  print(f"Unanimous votes: {unanimous}/{len(records)} ({unanimous / len(records):.1%})")
  print(f"Split votes (True and False both appeared): {split_votes}/{len(records)} ({split_votes / len(records):.1%})")
  print("Vote distributions:")
  for pattern, count in sorted(pattern_counter.items(), key=lambda x: (-x[1], x[0])):
    print(f"  True:{pattern[0]}, False:{pattern[1]} -> {count} questions")

print("\nExample split-vote cases at temperature = 1.0:")
for record in sc_temp_results[1.0]:
  if record['split_vote']:
    print(f"Q: {record['question']}")
    print(f"  True answer: {record['true_answer']}")
    print(f"  Votes: {record['votes']}")
    print(f"  Final prediction: {record['pred']}")
    print()


=== Temperature = 0 ===
Accuracy: 0.680
Unanimous votes: 50/50 (100.0%)
Split votes (True and False both appeared): 0/50 (0.0%)
Vote distributions:
  True:0, False:5 -> 37 questions
  True:5, False:0 -> 13 questions

=== Temperature = 0.5 ===
Accuracy: 0.640
Unanimous votes: 40/50 (80.0%)
Split votes (True and False both appeared): 10/50 (20.0%)
Vote distributions:
  True:0, False:5 -> 34 questions
  True:5, False:0 -> 6 questions
  True:4, False:1 -> 5 questions
  True:2, False:3 -> 2 questions
  True:3, False:2 -> 2 questions
  True:1, False:4 -> 1 questions

=== Temperature = 1.0 ===
Accuracy: 0.640
Unanimous votes: 37/50 (74.0%)
Split votes (True and False both appeared): 13/50 (26.0%)
Vote distributions:
  True:0, False:5 -> 31 questions
  True:5, False:0 -> 6 questions
  True:2, False:3 -> 5 questions
  True:4, False:1 -> 4 questions
  True:1, False:4 -> 3 questions
  True:3, False:2 -> 1 questions

Example split-vote cases at temperature = 1.0:
Q: Do the directors of The Matrix

# In Context Learning

## Loading the demonatrations

In [18]:
import json

with open("strategyqa_demo_questions_50.json", "r") as f:
    demos_dict = json.load(f)

print("Keys in the loaded dictionary:", demos_dict[0].keys())
print("Example Datapoint:\n", demos_dict[0])

Keys in the loaded dictionary: dict_keys(['qid', 'question', 'answer', 'facts', 'decomposition'])
Example Datapoint:
 {'qid': '09eaaf2d0f2a18d4b785', 'question': 'Would you find a tibia beside parsley on a holiday plate?', 'answer': True, 'facts': ['The tibia of a goat is eaten during Passover, a Jewish holiday', 'Parsley is served on a Passover seder plate beside the goat shank '], 'decomposition': ['How is Passover celebrated?', 'What part of a goat is eaten during #1?', 'Is parsley typically served on the same plate as #2?']}


## Creating an informative in-context prompts

In [19]:
def bulid_demo_prompt(datapoint):
  """
  You may use datapoint['question'], datapoint['answer']
  """
  answer = "True" if datapoint['answer'] else "False"
  prompt = f"Question: {datapoint['question']}\nAnswer: {answer}\n\n"
  return prompt

## Using the demonstartions for prediction

In [20]:
def icl_pred(question, num_demo_examples=10, demos=None):
  temperatue = 0
  demos_list_n = demos if demos is not None else demos_dict[:num_demo_examples]
  prompt = ""
  for datapoint in demos_list_n:
    prompt += bulid_demo_prompt(datapoint)
  prompt += f"Question: {question}\nAnswer:"
  pred = query_llama(prompt=prompt, temperature=temperatue)
  return pred


def demo_similarity(question, demo_question):
  words_q = set(question.lower().split())
  words_d = set(demo_question.lower().split())
  if not words_q or not words_d:
    return 0
  return len(words_q & words_d) / len(words_q | words_d)


def select_similar_demos(question, num_demo_examples=10):
  ranked = sorted(
    demos_dict,
    key=lambda d: demo_similarity(question, d['question']),
    reverse=True,
  )
  return ranked[:num_demo_examples]


def evaluate_icl_setting(name, pred_fn):
  correct = [evaluate(row['answer'], pred_fn(row['question'])) for _, row in df.iterrows()]
  acc = sum(correct) / len(correct)
  print(f"{name}: {acc:.3f}")
  return acc

In [21]:
df["icl_pred"] = df["question"].apply(lambda q: icl_pred(q))
icl_acc = df.apply(lambda row: evaluate(row['answer'], row['icl_pred']), axis=1).mean()
print(f"In Context Learning acc with Llama: {icl_acc}")

In Context Learning acc with Llama: 0.5


## 1.4.e ICL Analysis

Compare how the number of demonstrations, their order, and their similarity to the target question affect accuracy.

In [22]:
import random

print("=== Effect of number of demonstrations ===")
for n in [1, 3, 5, 10, 20]:
  evaluate_icl_setting(
    f"n={n}",
    lambda q, n=n: icl_pred(q, num_demo_examples=n),
  )

print("\n=== Effect of demonstration ordering (n=10) ===")
n = 10
evaluate_icl_setting(
  "original order",
  lambda q: icl_pred(q, demos=demos_dict[:n]),
)
evaluate_icl_setting(
  "reversed order",
  lambda q: icl_pred(q, demos=list(reversed(demos_dict[:n]))),
)
random.seed(42)
shuffled_demos = random.sample(demos_dict, n)
evaluate_icl_setting(
  "random order",
  lambda q: icl_pred(q, demos=shuffled_demos),
)

print("\n=== Effect of demonstration similarity (n=10) ===")
evaluate_icl_setting(
  "first 10 demos",
  lambda q: icl_pred(q, num_demo_examples=10),
)
evaluate_icl_setting(
  "10 most similar demos per question",
  lambda q: icl_pred(q, demos=select_similar_demos(q, 10)),
)

=== Effect of number of demonstrations ===
n=1: 0.500
n=3: 0.500
n=5: 0.500
n=10: 0.500
n=20: 0.640

=== Effect of demonstration ordering (n=10) ===
original order: 0.500
reversed order: 0.500
random order: 0.640

=== Effect of demonstration similarity (n=10) ===
first 10 demos: 0.500
10 most similar demos per question: 0.540


0.54

In [45]:
for _, row in df.iterrows():
  if evaluate(row['answer'], row['icl_pred']) == 0:
    print(f"Question: {row['question']}")
    print(f"  True answer:  {row['answer']}")
    print(f"  Model answer: {row['icl_pred']}")
    print()

Question: Do the directors of The Matrix advocate for transgender rights?
  True answer:  True
  Model answer: False

Question: Would it be difficult for Will Ferrell to win Empire Award for Best Newcomer?
  True answer:  True
  Model answer: False

Question: Does Orange County, California require airplanes to be quiet?
  True answer:  True
  Model answer: False

Question: If someone loves buffalo wings do they enjoy capsaicin?
  True answer:  True
  Model answer: False

Question: Would a Pict be confused by Old English?
  True answer:  True
  Model answer: False

Question: Is it hard to get a BLT in Casablanca?
  True answer:  True
  Model answer: False

Question: Does the land in close proximity to beaver dams suffer?
  True answer:  True
  Model answer: False

Question: Should a Celiac sufferer avoid spaghetti?
  True answer:  True
  Model answer: False

Question: Would the yearly precipitation on Snowdon submerge an upright bowling pin?
  True answer:  True
  Model answer: False

Q

# Chain of Thought

In [23]:
COT_SYSTEM_PROMPT = (
  "You are a helpful assistant solving multi-step yes/no questions. "
  "Think step by step and explain your reasoning before answering. "
  "After your reasoning, end with exactly one final line in this format: "
  "Final Answer: True or Final Answer: False"
)


def extract_cot_answer(response):
  for line in reversed(response.splitlines()):
    if 'final answer' in line.lower():
      ans = normalize_answer(line)
      if ans is not None:
        return ans
  return normalize_answer(response)


def cot_pred(question):
  temperatue = 0
  response = query_llama(
    prompt=question,
    temperature=temperatue,
    sys_prompt=COT_SYSTEM_PROMPT,
    max_tokens=1024,
  )
  pred = extract_cot_answer(response)
  if pred is None:
    pred = response
  return pred

In [24]:
df["cot_pred"] = df["question"].apply(lambda q: cot_pred(q))
cot_acc = df.apply(lambda row: evaluate(row['answer'], row['cot_pred']), axis=1).mean()
print(f"Chain of Thought acc with Llama: {cot_acc}")

Chain of Thought acc with Llama: 0.76


In [25]:
for _, row in df.iterrows():
  if evaluate(row['answer'], row['cot_pred']) == 0:
    print(f"Question: {row['question']}")
    print(f"  True answer:  {row['answer']}")
    print(f"  Model answer: {row['cot_pred']}")
    print()

Question: Would it be difficult for Will Ferrell to win Empire Award for Best Newcomer?
  True answer:  True
  Model answer: False

Question: Does Orange County, California require airplanes to be quiet?
  True answer:  True
  Model answer: False

Question: Is it hard to get a BLT in Casablanca?
  True answer:  True
  Model answer: False

Question: Is the Fibonacci number sequence longer than every number discovered in Pi?
  True answer:  True
  Model answer: False

Question: Could you drive a Rowe 550 to the 2008 Summer Olympics?
  True answer:  True
  Model answer: False

Question: Would a body builder prefer an elk burger over a beef burger?
  True answer:  True
  Model answer: False

Question: Does Post Malone have a fear of needles?
  True answer:  False
  Model answer: True

Question: Is Morocco an ideal location for water skiing?
  True answer:  False
  Model answer: True

Question: Would Achilles dominate Legolas in a hypothetical fight?
  True answer:  False
  Model answer: Tr

# ICL + CoT Combined

In [26]:
def bulid_cot_demo_prompt(datapoint):
  """
  You may use datapoint['question'], datapoint['answer'], datapoint['facts'], datapoint['decomposition']
  however you want.
  """
  answer = "True" if datapoint['answer'] else "False"
  facts_text = "\n".join(f"- {fact}" for fact in datapoint['facts'])
  steps_text = "\n".join(f"{i + 1}. {step}" for i, step in enumerate(datapoint['decomposition']))

  prompt = (
    f"Question: {datapoint['question']}\n"
    f"Facts:\n{facts_text}\n"
    f"Reasoning steps:\n{steps_text}\n"
    f"Final Answer: {answer}\n\n"
  )
  return prompt

In [27]:
COT_ICL_PROMPT = (
  "You are a helpful assistant solving multi-step yes/no questions. "
  "Study the solved examples to learn how to use facts and reasoning steps before answering. "
  "For the new question, think step by step, explain your reasoning, and end with exactly one line: "
  "Final Answer: True or Final Answer: False"
)

In [52]:
def icl_cot_pred(question, num_demo_examples=10):
  temperatue = 0
  prompt = ""
  for datapoint in demos_dict[:num_demo_examples]:
    prompt += bulid_cot_demo_prompt(datapoint)
  prompt += (
    f"Question: {question}\n"
    "Facts:\n"
    "Reasoning steps:\n"
    "Final Answer:"
  )

  response = query_llama(
    prompt=prompt,
    temperature=temperatue,
    sys_prompt=COT_ICL_PROMPT,
    max_tokens=1024,
  )
  pred = extract_cot_answer(response)
  if pred is None:
    pred = response
  return pred

In [53]:
df["icl_cot_pred"] = df["question"].apply(lambda q: icl_cot_pred(q))
icl_cot_acc = df.apply(lambda row: evaluate(row['answer'], row['icl_cot_pred']), axis=1).mean()
print(f"In Context Learning + Chain of Thought acc with Llama: {icl_cot_acc}")

In Context Learning + Chain of Thought acc with Llama: 0.66


In [54]:
comparison = {
  "Zero-shot": "zero_shot_pred",
  "ICL": "icl_pred",
  "CoT": "cot_pred",
  "ICL + CoT": "icl_cot_pred",
}

print("=== Method comparison (1.6.e) ===")
for name, col in comparison.items():
  if col in df.columns:
    acc = df.apply(lambda row: evaluate(row['answer'], row[col]), axis=1).mean()
    print(f"{name}: {acc:.3f}")

for _, row in df.iterrows():
  if evaluate(row['answer'], row['icl_cot_pred']) == 0:
    print(f"\nQuestion: {row['question']}")
    print(f"  True answer:  {row['answer']}")
    print(f"  Model answer: {row['icl_cot_pred']}")

=== Method comparison (1.6.e) ===
Zero-shot: 0.680
ICL: 0.500
CoT: 0.760
ICL + CoT: 0.660

Question: Would it be difficult for Will Ferrell to win Empire Award for Best Newcomer?
  True answer:  True
  Model answer: False

Question: Does Orange County, California require airplanes to be quiet?
  True answer:  True
  Model answer: False

Question: Would a Pict be confused by Old English?
  True answer:  True
  Model answer: False

Question: Is it hard to get a BLT in Casablanca?
  True answer:  True
  Model answer: False

Question: Is the Fibonacci number sequence longer than every number discovered in Pi?
  True answer:  True
  Model answer: False

Question: Would an owl monkey enjoy a strawberry?
  True answer:  True
  Model answer: False

Question: Is tobacco use made to seem enjoyable in Alice's Adventures in Wonderland?
  True answer:  True
  Model answer: False

Question: Was a person sold a Creative Commons License for Boticelli's The Birth of Venus ripped off?
  True answer:  Tr